# Notebook 10 — TGF-β Pathway Activity Scoring and TME Rewiring Analysis

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/10_communication.h5ad`  
**Output:** `data/processed/11_pathway_scored.h5ad`, pathway UMAPs, violin plots, TME rewiring volcano

---

## Scientific Background

TGF-β (Transforming Growth Factor beta) signaling is a central mediator of tumour invasion and immunosuppression. Wan et al. (2025, *Ophthalmology*) identified that the **CP4 cone precursor subcluster** — specifically enriched in extraocular samples — shows the highest TGF-β ligand expression (TGFB1, TGFB2).

This notebook quantifies TGF-β pathway activity across the integrated atlas to:

1. **Confirm TGF-β upregulation** in extraocular vs. intraocular tumour cells
2. **Identify cell type sources and responders**: Which cells produce TGF-β (autocrine) and which respond (paracrine)?
3. **Correlate TGF-β activity with**:
   - Fate probability of the invasive terminal state (notebook 08)
   - RB Subtype 2 / stemness score (notebook 06)
   - CNV instability load (notebook 05)
4. **Score TME-relevant pathways** alongside TGF-β for a comprehensive invasion landscape

## Method — decoupleR with PROGENy

**PROGENy** (Schubert et al. 2018, *Nat Commun*) provides gene expression footprints of 14 canonical signalling pathways derived from **perturbation experiments** (not correlation data), making scores mechanistically interpretable.

**decoupleR** (Badia-i-Mompel et al. 2022, *Bioinformatics*) applies the **weighted mean (wmean)** estimator — the recommended method for PROGENy.

**References:**  
- Wan W et al. *Ophthalmology* 2025. https://doi.org/10.1016/j.ophtha.2025.01.011  
- Schubert M et al. *Nat Commun* 2018. https://doi.org/10.1038/s41467-017-02391-6  
- Badia-i-Mompel P et al. *Bioinformatics* 2022. https://doi.org/10.1093/bioadv/btac016

## TGF-β Canonical Target Genes (Manual Score)

In [ ]:
# PROGENy pathways to score
PROGENY_PATHWAYS = [
    "TGFb",     # TGF-beta signaling — primary focus
    "VEGF",     # Angiogenesis
    "Hypoxia",  # HIF1A targets
    "MAPK",     # ERK/RAF/RAS
    "PI3K",     # AKT/mTOR
    "p53",      # Tumour suppressor
    "JAK-STAT", # Cytokine signaling
    "WNT",      # Beta-catenin / dedifferentiation
    "NFkB",     # Inflammation
    "EGFR",     # EGF receptor signaling
    "Trail",    # Apoptosis
    "TNFa",     # TNF-alpha (inflammatory)
]
PROGENY_TOP_N = 100  # Top N genes per pathway for PROGENy

# TGF-beta canonical target genes (manual validation score)
TGFB_CANONICAL_TARGETS = [
    # SMAD-dependent targets
    "SMAD7", "SNAI1", "SNAI2", "TWIST1", "ZEB1", "ZEB2",
    "CDH2", "VIM", "FN1", "MMP2", "MMP9", "CTGF", "CYR61",
    "TGFBI", "ITGB6", "SERPINE1", "COL1A1", "COL1A2", "ACTA2",
    # TGF-beta ligands / receptors
    "TGFB1", "TGFB2", "TGFB3", "TGFBR1", "TGFBR2", "TGFBR3",
    "SMAD2", "SMAD3", "SMAD4",
    # EMT markers
    "CDH1",  # E-cadherin (decreases in EMT)
    "VIM",   # Vimentin (increases in EMT)
]

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "10_communication.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "11_pathway_scored.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Communication-Annotated Atlas

In [ ]:
print("Step 1 — Loading communication-annotated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")

## Step 2 — PROGENy Pathway Scoring via decoupleR

PROGENy scores estimate the **activity** of signalling pathways, not just the expression of individual pathway genes. This is more interpretable biologically: a pathway can be active even if individual genes have variable expression, as long as the ensemble footprint is coherent.

The `wmean` method computes a **weighted mean** of top-100 target gene expressions per pathway, with weights derived from perturbation experiments.

**Note:** Requires `pip install decoupler`. If not installed, falls back to manual TGF-β scoring.

In [ ]:
print("Step 2 — PROGENy pathway activity scoring (decoupleR wmean)")

try:
    import decoupler as dc

    print("  Loading PROGENy network (human, top 100 genes per pathway)...")
    progeny_net = dc.get_progeny(organism="human", top=PROGENY_TOP_N)
    print(f"  PROGENy network: {len(progeny_net):,} gene-pathway edges")

    print("  Running decoupleR wmean estimator...")
    dc.run_wmean(
        mat=adata,
        net=progeny_net,
        source="source",
        target="target",
        weight="weight",
        times=100,  # permutations for p-value
        min_n=5,    # min genes per pathway
        use_raw=False,
    )

    if "wmean_estimate" in adata.obsm:
        acts = pd.DataFrame(
            adata.obsm["wmean_estimate"],
            index=adata.obs_names,
        )
        for pathway in acts.columns:
            adata.obs[f"PROGENy_{pathway}"] = acts[pathway].values
        print(f"  Scored {len(acts.columns)} pathways via PROGENy wmean.")
        print(f"  Pathways: {acts.columns.tolist()}")
    else:
        print("  WARNING: wmean_estimate not in adata.obsm")

except ImportError:
    print("  decoupleR not installed. Install: pip install decoupler")
    print("  Falling back to manual TGF-beta scoring only.")
except Exception as e:
    print(f"  PROGENy scoring failed: {e}")

## Step 3 — Manual TGF-β Signature Score

A manual score using canonical SMAD-dependent and EMT target genes serves as:
1. A **fallback** when decoupleR is unavailable
2. A **validation** of the PROGENy TGF-β score — both should be concordant (r > 0.5)

Uses `sc.tl.score_genes()` with expression-bin-matched control genes.

In [ ]:
print("Step 3 — Manual TGF-beta signature score")

present = [g for g in TGFB_CANONICAL_TARGETS if g in adata.var_names]
absent  = [g for g in TGFB_CANONICAL_TARGETS if g not in adata.var_names]
print(f"  Genes present: {len(present)}/{len(TGFB_CANONICAL_TARGETS)} (missing: {absent[:5]})")

sc.tl.score_genes(
    adata,
    gene_list=present,
    score_name="TGFb_manual_score",
    ctrl_size=50,
    n_bins=25,
)
print(f"  TGF-beta manual score: mean={adata.obs['TGFb_manual_score'].mean():.4f}, "
      f"std={adata.obs['TGFb_manual_score'].std():.4f}")

# Validate concordance with PROGENy score
if "PROGENy_TGFb" in adata.obs.columns:
    r, p = stats.pearsonr(
        adata.obs["PROGENy_TGFb"].fillna(0),
        adata.obs["TGFb_manual_score"].fillna(0),
    )
    print(f"  Correlation PROGENy_TGFb vs manual: r={r:.3f}, p={p:.2e}")
    if abs(r) < 0.5:
        print("  WARNING: Low concordance — check PROGENy gene availability")

## Step 4 — Pathway UMAP Grid

**What to look for:**
- **TGF-β**: High scores in cone precursor/invasive clusters, especially in extraocular cells
- **VEGF**: High in tumour vasculature-adjacent cells or hypoxic regions
- **Hypoxia**: High in intraocular core (avascular microenvironment)
- **WNT**: High in dedifferentiated Subtype 2 cells
- **p53**: Low in tumour cells with RB1/MDM4 pathway dysregulation

In [ ]:
pathway_cols = [c for c in adata.obs.columns
                if c.startswith("PROGENy_") or c == "TGFb_manual_score"]

if pathway_cols:
    focus = ["PROGENy_TGFb", "PROGENy_VEGF", "PROGENy_Hypoxia",
             "PROGENy_MAPK", "PROGENy_WNT", "TGFb_manual_score"]
    cols  = [c for c in focus if c in pathway_cols][:6]
    if not cols:
        cols = pathway_cols[:6]

    ncols = 3
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten()

    for ax, col in zip(axes, cols):
        sc.pl.umap(
            adata, color=col, ax=ax, show=False, frameon=False,
            size=2, cmap="RdBu_r", vcenter=0,
            title=col.replace("PROGENy_", "").replace("_manual_score", " (manual)"),
        )
    for ax in axes[len(cols):]:
        ax.set_visible(False)

    plt.suptitle("Pathway activity scores (PROGENy / decoupleR)",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "pathway_scores_umap.pdf", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("  No pathway score columns found. Run Step 2 first.")

## Step 5 — TGF-β Activity by Cell Type and Disease Stage

**Key hypothesis**: TGF-β activity is higher in extraocular tumour cone precursor cells (Wan et al. 2025).

Each cell type is shown separately to distinguish:
- **Cell-autonomous**: tumour-intrinsic TGF-β upregulation
- **Paracrine**: stromal cells contributing TGF-β to the TME

In [ ]:
tgfb_col  = "PROGENy_TGFb" if "PROGENy_TGFb" in adata.obs.columns else "TGFb_manual_score"
stage_col = "disease_stage" if "disease_stage" in adata.obs.columns else "dataset"

if tgfb_col in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    split = stage_col if adata.obs[stage_col].nunique() <= 3 else None
    sc.pl.violin(
        adata, keys=tgfb_col, groupby="cell_type_broad",
        split=split, ax=ax, show=False, rotation=30,
    )
    ax.set_title(
        f"TGF-beta pathway activity by cell type\n"
        f"(split by {stage_col}; higher = more active)",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / "tgfb_score_by_celltype_stage.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 6 — Pathway Heatmap by Cell Type

Shows mean pathway activity per cell type, revealing which pathways are most active in each cell population. This is a comprehensive view of the signalling landscape.

In [ ]:
pathway_cols = [c for c in adata.obs.columns if c.startswith("PROGENy_")]

if pathway_cols:
    ct_mean = adata.obs.groupby("cell_type_broad")[pathway_cols].mean()
    ct_mean.columns = [c.replace("PROGENy_", "") for c in ct_mean.columns]

    fig, ax = plt.subplots(figsize=(len(pathway_cols) * 0.8 + 2,
                                     len(ct_mean) * 0.6 + 2))
    vmax = ct_mean.abs().max().max()
    im = ax.imshow(
        ct_mean.values, aspect="auto", cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
    )
    ax.set_xticks(range(len(ct_mean.columns)))
    ax.set_xticklabels(ct_mean.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(ct_mean.index)))
    ax.set_yticklabels(ct_mean.index, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.6, label="Mean pathway activity")
    ax.set_title("PROGENy pathway activities per cell type\n(Red = active, Blue = inactive)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "pathway_heatmap_by_celltype.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 7 — Differential Pathway Analysis

**Statistical test**: Mann-Whitney U (non-parametric; pathway scores are right-skewed)  
**Multiple testing correction**: Benjamini-Hochberg FDR  
**Significance threshold**: FDR < 0.05 AND |log2FC| > 0.5

**Expected result** (Wan et al. 2025): TGF-β significantly higher in extraocular samples.

In [ ]:
print("Step 7 — Differential pathway analysis (intraocular vs. extraocular)")

pathway_cols = ([c for c in adata.obs.columns if c.startswith("PROGENy_")]
                + ["TGFb_manual_score"])
group_col = "disease_stage" if "disease_stage" in adata.obs.columns else "dataset"
group_a   = "intraocular" if "disease_stage" in adata.obs.columns else "GSE168434"
group_b   = "extraocular" if "disease_stage" in adata.obs.columns else "GSE249995"

if group_col not in adata.obs.columns:
    print(f"  Column '{group_col}' not found — skipping differential analysis")
    diff_df = pd.DataFrame()
else:
    mask_a = adata.obs[group_col].str.contains(group_a, case=False, na=False)
    mask_b = adata.obs[group_col].str.contains(group_b, case=False, na=False)
    print(f"  {group_a}: {mask_a.sum():,} cells | {group_b}: {mask_b.sum():,} cells")

    rows = []
    for col in pathway_cols:
        if col not in adata.obs.columns:
            continue
        a_vals = adata.obs.loc[mask_a, col].dropna().values
        b_vals = adata.obs.loc[mask_b, col].dropna().values
        if len(a_vals) < 10 or len(b_vals) < 10:
            continue
        stat, pval = stats.mannwhitneyu(a_vals, b_vals, alternative="two-sided")
        mean_a = a_vals.mean()
        mean_b = b_vals.mean()
        lfc    = np.log2((mean_b + 1e-6) / (mean_a + 1e-6))
        rows.append({
            "pathway": col, "group_a_mean": round(mean_a, 5),
            "group_b_mean": round(mean_b, 5),
            "log2fc": round(lfc, 4), "mwu_stat": round(stat, 1), "p_value": pval,
        })

    if rows:
        from scipy.stats import false_discovery_control
        diff_df = pd.DataFrame(rows)
        diff_df["p_adj"] = false_discovery_control(diff_df["p_value"].values)
        diff_df["significant"] = (diff_df["p_adj"] < 0.05) & (diff_df["log2fc"].abs() > 0.5)
        diff_df = diff_df.sort_values("log2fc", ascending=False)
        diff_df.to_csv(TAB_DIR / "tgfb_differential_analysis.csv", index=False)
        print("  Differential pathway results:")
        print(diff_df[["pathway", "log2fc", "p_adj", "significant"]].to_string())
        sig = diff_df[diff_df["significant"]]
        print(f"\n  Significantly rewired pathways: {sig['pathway'].tolist()}")
    else:
        diff_df = pd.DataFrame()
        print("  No pathway columns available for testing")

## Step 8 — TME Rewiring Volcano Plot

Shows which pathways are significantly more active in extraocular vs. intraocular tumours.

**Expected:**
- **Upper right** (high log2FC, low adj p): TGF-β, VEGF — invasion-promoting
- **Upper left** (negative log2FC): p53 — loss of tumour suppression in aggressive RB
- **Grey points**: pathways not significantly rewired

In [ ]:
if not diff_df.empty:
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = diff_df["significant"].map({True: "#E84C4C", False: "#AAAAAA"})
    ax.scatter(
        diff_df["log2fc"],
        -np.log10(diff_df["p_adj"] + 1e-300),
        s=60, c=colors, edgecolors="none", alpha=0.8,
    )
    for _, row in diff_df[diff_df["significant"]].iterrows():
        name = row["pathway"].replace("PROGENy_", "")
        ax.annotate(
            name,
            xy=(row["log2fc"], -np.log10(row["p_adj"] + 1e-300)),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
    ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=1)
    ax.axvline(0, color="grey", linestyle="-", linewidth=0.5)
    ax.set_xlabel("log2 FC (extraocular / intraocular)", fontsize=10)
    ax.set_ylabel("-log10(FDR)", fontsize=10)
    ax.set_title(
        "TME pathway rewiring: extraocular vs. intraocular\n"
        "(red = FDR < 0.05 and |log2FC| > 0.5)",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / "tme_rewiring_comparison.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 9 — TGF-β Correlation Analysis

Testing three mechanistic hypotheses:
1. **TGF-β ↔ RB Subtype 2**: TGF-β drives dedifferentiation
2. **TGF-β ↔ Invasive fate probability**: TGF-β promotes invasive fate commitment
3. **TGF-β ↔ CNV instability**: Genomically unstable cells activate TGF-β

In [ ]:
tgfb_col = "PROGENy_TGFb" if "PROGENy_TGFb" in adata.obs.columns else "TGFb_manual_score"

if tgfb_col in adata.obs.columns:
    correlates = [("score_RB_subtype2_stemness", "RB Subtype 2 score"),
                  ("cnv_load", "CNV instability load")]
    fate_cols = [c for c in adata.obs.columns if c.startswith("fate_prob_")]
    for fc in fate_cols[:2]:
        correlates.append((fc, fc.replace("fate_prob_", "P(-> ")))

    valid = [(c, l) for c, l in correlates if c in adata.obs.columns]
    if valid:
        ncols = min(3, len(valid))
        nrows = int(np.ceil(len(valid) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
        axes = np.array(axes).flatten() if len(valid) > 1 else [axes]

        for ax_idx, (col, label) in enumerate(valid):
            ax = axes[ax_idx]
            x = adata.obs[tgfb_col].values
            y = adata.obs[col].values
            idx = np.random.choice(len(x), size=min(5000, len(x)), replace=False)
            r_p, _   = stats.pearsonr(x[idx], y[idx])
            r_s, p_s = stats.spearmanr(x[idx], y[idx])
            ax.scatter(x[idx], y[idx], s=1, alpha=0.3, color="#4C9BE8", rasterized=True)
            ax.set_xlabel("TGF-beta pathway activity", fontsize=9)
            ax.set_ylabel(label, fontsize=9)
            ax.set_title(f"r_P={r_p:.3f}, r_S={r_s:.3f} (p={p_s:.1e})", fontsize=8)

        for ax in axes[len(valid):]:
            ax.set_visible(False)

        plt.suptitle("TGF-beta activity correlates with invasion markers",
                     fontsize=11, fontweight="bold", y=1.01)
        plt.tight_layout()
        fig.savefig(FIG_DIR / "tgfb_correlation_subtype_fate.pdf", dpi=200, bbox_inches="tight")
        plt.show()

## Step 10 — Save Pathway Score Table and Final Atlas

In [ ]:
# Save pathway score table
meta_cols  = ["sample_id", "dataset", "cell_type_broad", "rb_subtype",
              "cnv_load", "is_tumour_cnv"]
meta_cols  = [c for c in meta_cols if c in adata.obs.columns]
score_cols = [c for c in adata.obs.columns
              if c.startswith("PROGENy_") or c == "TGFb_manual_score"]
adata.obs[meta_cols + score_cols].to_csv(TAB_DIR / "progeny_pathway_scores.csv")
print(f"Saved pathway scores -> {TAB_DIR / 'progeny_pathway_scores.csv'}")

# Store differential analysis in uns
if not diff_df.empty:
    adata.uns["progeny_diff"] = diff_df.to_dict()

# Save final atlas
print(f"\nSaving final pathway-scored atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added:")
print("    .obs['PROGENy_*']        : 14 pathway activity scores")
print("    .obs['TGFb_manual_score']: canonical TGF-beta target gene score")
print("    .uns['progeny_diff']     : differential pathway analysis")
print("")
print("  PIPELINE COMPLETE")
print("  Final atlas: 11_pathway_scored.h5ad")
print("  Contains: QC -> normalization -> scVI integration -> annotation")
print("            -> CNV -> subtype scoring -> RNA velocity -> CellRank")
print("            -> L-R communication -> TGF-beta pathway scoring")